In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [ ]:
import json

file_path = "/home/user/Downloads/bayut_transactions_rented_2026_11_weekly01.json"


def stream_json_objects(file_path):
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        buffer = ""

        while True:
            chunk = f.read(1024 * 1024)  # 1 MB at a time

            if not chunk:
                break

            buffer += chunk

            while True:
                buffer = buffer.lstrip()

                if not buffer:
                    break

                try:
                    record, index = decoder.raw_decode(buffer)
                    yield record
                    buffer = buffer[index:]

                except json.JSONDecodeError:
                    break


# Test reading the file
count = 0

for record in stream_json_objects(file_path):
    count += 1

    if count <= 5:
        print(f"Record {count}:")
        print(record)
        print()

    if count % 100000 == 0:
        print(f"Processed {count:,} records")

print(f"\nTotal records: {count:,}")

In [2]:
import json
import re
import hashlib
import pandas as pd
from collections import Counter
from urllib.parse import urlsplit


file_path = "/home/user/Downloads/dubizzle_uae_2026_11_01.json"


# ==================================================
# 1. STREAM JSON RECORDS
# ==================================================
def stream_json_objects(file_path):
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        buffer = ""

        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            buffer += chunk

            while True:
                buffer = buffer.lstrip()

                if not buffer:
                    break

                try:
                    record, index = decoder.raw_decode(buffer)
                    yield record
                    buffer = buffer[index:]

                except json.JSONDecodeError:
                    break


# ==================================================
# 2. EXPECTED COLUMNS
# Edit this list if your requirement has more fields.
# ==================================================
expected_fields = {
url, reference_number, id, broker_display_name, broker, category, category_url, title, property_type, sub_category_1, sub_category_2, depth, description, location, price, currency, price_per, bedrooms, bathrooms, furnished, rera_permit_number, dtcm_licence, scraped_ts, amenities, details, agent_name, number_of_photos, user_id, phone_number, date, iteration_number, published_at, completion_status, ded_license_number, rera_registration_number, latitude, longitude, 
package_type, last_update, listed_by, verified, trakheesi_url
}


# ==================================================
# 3. INITIALIZE CHECKS
# ==================================================
record_count = 0

seen_records = set()

value_count_fields = [
    "scraped_ts",
    "iteration_number",
    "date",
    "category",
    "sub_category_1",
    "completion_status",
    "package_type",
    "currency",
    "verified",
    "published_at",
    "price_per",
    "furnished"
]

value_counts = {
    field: Counter() for field in value_count_fields
}

missing_counts = Counter()

whitespace_examples = {}
invalid_url_examples = []

duplicate_count = 0
broker_mismatch_count = 0
broker_case_count = 0

broker_mismatch_example = None
broker_case_example = None

actual_fields = set()

# Count missing/null/blank values in bedroom and bathroom fields
bedroom_fields = ["bedrooms", "bedroom"]
bathroom_fields = ["bathrooms", "bathroom"]


# ==================================================
# 4. PROCESS RECORDS
# ==================================================
for record_no, record in enumerate(
    stream_json_objects(file_path), start=1
):

    record_count += 1

    if not isinstance(record, dict):
        continue

    if record_count == 1:
        actual_fields = set(record.keys())

    # ----------------------------------------------
    # A. DATASET UNIQUENESS
    # ----------------------------------------------
    fingerprint = hashlib.sha256(
        json.dumps(
            record,
            sort_keys=True,
            ensure_ascii=False,
            default=str
        ).encode("utf-8")
    ).hexdigest()

    if fingerprint in seen_records:
        duplicate_count += 1
    else:
        seen_records.add(fingerprint)

    # ----------------------------------------------
    # B. VALUE COUNTS
    # ----------------------------------------------
    for field in value_count_fields:

        value = record.get(field)

        if value is None:
            value_counts[field]["<NULL>"] += 1

        elif isinstance(value, str) and value.strip() == "":
            value_counts[field]["<BLANK>"] += 1

        elif isinstance(value, (dict, list)):
            value_counts[field][str(value)] += 1

        else:
            value_counts[field][str(value)] += 1

    # ----------------------------------------------
    # C. WHITESPACE CHECK
    # One example per field only
    # ----------------------------------------------
    for field, value in record.items():

        if field in whitespace_examples:
            continue

        if not isinstance(value, str):
            continue

        if (
            value != value.strip()
            or re.search(r" {2,}", value)
            or re.search(r"[\t\r\n]", value)
        ):
            whitespace_examples[field] = {
                "record_no": record_no,
                "field": field,
                "url": record.get("url", "")
            }

    # ----------------------------------------------
    # D. BEDROOM AND BATHROOM MISSING VALUES
    # ----------------------------------------------
    for field in bedroom_fields + bathroom_fields:

        if field in record:
            value = record[field]

            if (
                value is None
                or (isinstance(value, str) and value.strip() == "")
            ):
                missing_counts[field] += 1

    # ----------------------------------------------
    # E. INVALID URL CHECK
    # ----------------------------------------------
    url = record.get("url")

    if not (
        isinstance(url, str)
        and url.strip()
        and urlsplit(url.strip()).scheme in ("http", "https")
        and urlsplit(url.strip()).netloc
    ):
        if len(invalid_url_examples) < 5:
            invalid_url_examples.append({
                "record_no": record_no,
                "url": url
            })

    # ----------------------------------------------
    # F. BROKER CONSISTENCY
    # ----------------------------------------------
    broker = record.get("broker")
    broker_display = record.get("broker_display_name")

    broker_value = str(broker).strip() if broker is not None else ""
    display_value = (
        str(broker_display).strip()
        if broker_display is not None else ""
    )

    # Compare values when at least one field is populated
    if broker_value or display_value:

        if broker_value != display_value:
            broker_mismatch_count += 1

            if broker_mismatch_example is None:
                broker_mismatch_example = {
                    "record_no": record_no,
                    "url": url
                }

    # Check broker uppercase
    if broker_value and broker_value != broker_value.upper():
        broker_case_count += 1

        if broker_case_example is None:
            broker_case_example = {
                "record_no": record_no,
                "url": url
            }

    if record_count % 100000 == 0:
        print(f"Processed {record_count:,} records")


# ==================================================
# 5. DATASET UNIQUENESS RESULT
# ==================================================
print("\n========== DATASET UNIQUENESS ==========")

print(f"Total records: {record_count:,}")
print(f"Unique records: {len(seen_records):,}")
print(f"Duplicate records: {duplicate_count:,}")

if duplicate_count == 0:
    print("✓ Dataset is unique")
else:
    print("✗ Duplicate records found")


# ==================================================
# 6. COLUMN ADDITION / MISSING CHECK
# ==================================================
print("\n========== COLUMN CHECK ==========")

missing_columns = expected_fields - actual_fields
additional_columns = actual_fields - expected_fields

print("\nMissing columns:")
if missing_columns:
    for field in sorted(missing_columns):
        print("✗", field)
else:
    print("✓ No missing columns")

print("\nAdditional columns:")
if additional_columns:
    for field in sorted(additional_columns):
        print("+", field)
else:
    print("✓ No additional columns")


# ==================================================
# 7. VALUE COUNTS
# ==================================================
print("\n========== VALUE COUNTS ==========")

for field, counts in value_counts.items():

    print(f"\n--- {field} ---")

    if counts:
        for value, count in counts.most_common():
            print(f"{value}: {count:,}")
    else:
        print("No values found")


# ==================================================
# 8. BEDROOM / BATHROOM MISSING VALUES
# ==================================================
print("\n========== BEDROOM / BATHROOM MISSING VALUES ==========")

for field in bedroom_fields + bathroom_fields:

    if field in actual_fields:
        print(f"{field}: {missing_counts[field]:,} missing/null/blank")


# ==================================================
# 9. WHITESPACE ISSUES
# One example per field
# ==================================================
print("\n========== WHITESPACE ISSUES ==========")

if whitespace_examples:
    whitespace_result = pd.DataFrame(
        whitespace_examples.values()
    )
    display(whitespace_result)
else:
    print("✓ No whitespace issues found")


# ==================================================
# 10. INVALID URL ISSUES
# ==================================================
print("\n========== INVALID URL CHECK ==========")

print(
    "Invalid URL examples shown (maximum 5):",
    len(invalid_url_examples)
)

if invalid_url_examples:
    display(pd.DataFrame(invalid_url_examples))
else:
    print("✓ No invalid URLs found in the sample check")


# ==================================================
# 11. BROKER CHECK
# ==================================================
print("\n========== BROKER CHECK ==========")

print(f"Broker value mismatches: {broker_mismatch_count:,}")

if broker_mismatch_example:
    print("Mismatch example:", broker_mismatch_example)
else:
    print("✓ Broker values match")

print(f"\nBroker not uppercase: {broker_case_count:,}")

if broker_case_example:
    print("Not uppercase example:", broker_case_example)
else:
    print("✓ Broker values are uppercase")

NameError: name 'url' is not defined

In [2]:
import re
import json
import hashlib
import random
from datetime import datetime
from collections import Counter

file_path = "/home/user/Downloads/emlakjet_tur_2026_11.json"

# --------------------------------------------------
# Settings
# --------------------------------------------------
date_fields = ["scraped_ts", "date"]

empty_fields = [
    "reference_number",
    "furnished",
    "price_per",
    "rera_permit_number",
    "dtcm_licence"
]

allowed_currencies = {"TL", "GBP", "USD", "EUR"}

issue_counts = Counter()
issue_examples = {}
seen_records = set()
random_records = []

random_generator = random.Random(42)
record_count = 0


# --------------------------------------------------
# Helper functions
# --------------------------------------------------
def is_empty(value):
    return (
        value is None
        or (isinstance(value, str) and value.strip() == "")
        or (isinstance(value, (list, dict)) and len(value) == 0)
    )


def add_issue(issue, record_no, record):
    issue_counts[issue] += 1

    # Keep only five examples per issue
    if issue not in issue_examples:
        issue_examples[issue] = []

    if len(issue_examples[issue]) < 5:
        issue_examples[issue].append({
            "record_no": record_no,
            "profile_url": record.get("profile_url", "")
        })


def get_date_day(value):
    if is_empty(value):
        return None

    # Try common date formats
    value = str(value).strip()

    formats = [
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S",
        "%d-%m-%Y",
        "%d/%m/%Y",
        "%Y/%m/%d"
    ]

    for fmt in formats:
        try:
            return datetime.strptime(value[:19], fmt).day
        except ValueError:
            continue

    # Handle ISO timestamps with timezone or milliseconds
    try:
        return datetime.fromisoformat(
            value.replace("Z", "+00:00")
        ).day
    except ValueError:
        return None


# --------------------------------------------------
# Read and validate the JSON records
# --------------------------------------------------
for record_no, record in enumerate(
    stream_json_objects(file_path), start=1
):
    record_count += 1

    if not isinstance(record, dict):
        add_issue("Record is not a JSON object", record_no, {})
        continue

    # 1. Dataset uniqueness: detect duplicate full records
    fingerprint = hashlib.sha256(
        json.dumps(
            record,
            sort_keys=True,
            ensure_ascii=False,
            default=str
        ).encode("utf-8")
    ).hexdigest()

    if fingerprint in seen_records:
        add_issue("Duplicate record", record_no, record)
    else:
        seen_records.add(fingerprint)

    # 2. scraped_ts and date: day must be 21–31
    for field in date_fields:
        value = record.get(field)

        if is_empty(value):
            add_issue(f"{field}: empty", record_no, record)
        else:
            day = get_date_day(value)

            if day is None:
                add_issue(f"{field}: invalid date", record_no, record)
            elif day not in range(21, 32):
                add_issue(f"{field}: day outside 21–31", record_no, record)

    # 3. iteration_number must follow YYYY_MM
    iteration = record.get("iteration_number")

    if is_empty(iteration) or not re.fullmatch(
        r"\d{4}_(0[1-9]|1[0-2])", str(iteration).strip()
    ):
        add_issue(
            "iteration_number: invalid YYYY_MM format",
            record_no, record
        )

    # 4. Currency: allow TL, GBP, USD, EUR
    currency = record.get("currency")

    if is_empty(currency):
        add_issue("currency: empty", record_no, record)
    elif str(currency).strip().upper() not in allowed_currencies:
        add_issue("currency: unexpected value", record_no, record)

    # 5. broker and broker_display_name
    broker = record.get("broker")
    broker_display = record.get("broker_display_name")

    if not is_empty(broker):
        if str(broker) != str(broker).upper():
            add_issue("broker: not uppercase", record_no, record)

    if not is_empty(broker) or not is_empty(broker_display):
        if str(broker or "").strip() != str(
            broker_display or ""
        ).strip():
            add_issue(
                "broker and broker_display_name: mismatch",
                record_no, record
            )

    # 6. category must be rent or sale
    category = record.get("category")

    if is_empty(category) or str(category).strip().lower() not in {
        "rent", "sale"
    }:
        add_issue("category: invalid value", record_no, record)

    # 7. Check specified fields for empty values
    for field in empty_fields:
        if is_empty(record.get(field)):
            add_issue(f"{field}: empty", record_no, record)

    # 8. Random sample of 10 records (reservoir sampling)
    if len(random_records) < 10:
        random_records.append((record_no, record))
    else:
        position = random_generator.randint(1, record_count)

        if position <= 10:
            random_records[position - 1] = (record_no, record)

    if record_count % 100000 == 0:
        print(f"Processed {record_count:,} records")


# --------------------------------------------------
# Results
# --------------------------------------------------
print("\n========== DATASET SUMMARY ==========")
print(f"Total records: {record_count:,}")
print(f"Unique records: {len(seen_records):,}")

if record_count == len(seen_records):
    print("Dataset uniqueness: ✓ PASS")
else:
    print("Dataset uniqueness: ✗ FAIL")
    print(f"Duplicate records: {record_count - len(seen_records):,}")

print("\n========== ISSUE COUNTS ==========")

if not issue_counts:
    print("✓ No issues found")
else:
    for issue, count in issue_counts.most_common():
        print(f"✗ {issue}: {count:,}")

print("\n========== ISSUE EXAMPLES ==========")

for issue, examples in issue_examples.items():
    print(f"\n{issue}")
    for example in examples:
        print(
            f"Record {example['record_no']} | "
            f"Profile URL: {example['profile_url']}"
        )

print("\n========== 10 RANDOM RECORDS ==========")

for record_no, record in random_records:
    print(f"\nRecord {record_no}:")
    print(json.dumps(record, ensure_ascii=False, indent=2))

Processed 100,000 records
Processed 200,000 records
Processed 300,000 records

========== DATASET SUMMARY ==========
Total records: 300,422
Unique records: 300,422
Dataset uniqueness: ✓ PASS

========== ISSUE COUNTS ==========
✗ reference_number: empty: 300,422
✗ furnished: empty: 300,422
✗ price_per: empty: 300,422
✗ rera_permit_number: empty: 300,422
✗ dtcm_licence: empty: 300,422
✗ broker and broker_display_name: mismatch: 130,387
✗ currency: empty: 370
✗ currency: unexpected value: 97

========== ISSUE EXAMPLES ==========

reference_number: empty
Record 1 | Profile URL: 
Record 2 | Profile URL: 
Record 3 | Profile URL: 
Record 4 | Profile URL: 
Record 5 | Profile URL: 

furnished: empty
Record 1 | Profile URL: 
Record 2 | Profile URL: 
Record 3 | Profile URL: 
Record 4 | Profile URL: 
Record 5 | Profile URL: 

price_per: empty
Record 1 | Profile URL: 
Record 2 | Profile URL: 
Record 3 | Profile URL: 
Record 4 | Profile URL: 
Record 5 | Profile URL: 

rera_permit_number: empty
Recor

In [3]:
from collections import Counter

currency_counts = Counter()

for record in stream_json_objects(file_path):
    if isinstance(record, dict):
        currency = record.get("currency")
        currency_counts[str(currency) if currency is not None else "None"] += 1

for currency, count in currency_counts.most_common():
    print(f"{currency}: {count:,}")

TL: 299,955
: 370
£: 90
$: 4
€: 3


In [3]:
import re
import pandas as pd

file_path = "/home/user/Downloads/dubizzle_uae_2026_11_01.json"

# Store only the first whitespace issue found in each field
issues = {}

for record_no, record in enumerate(
    stream_json_objects(file_path), start=1
):
    if not isinstance(record, dict):
        continue

    profile_url = record.get("profile_url", "")

    for field, value in record.items():

        if field in issues or not isinstance(value, str):
            continue

        # Check leading/trailing spaces or multiple consecutive spaces
        if (
            value != value.strip()
            or re.search(r" {2,}", value)
        ):
            issues[field] = {
                "field": field,
                "profile_url": profile_url,
                "record_no": record_no
            }

# Display one issue per field, without showing the value
result = pd.DataFrame(issues.values())

if not result.empty:
    print(result.to_string(index=False))
else:
    print("No whitespace issues found.")

              field profile_url  record_no
        description                      2
              title                     12
   reference_number                   4263
         agent_name                   5041
             broker                   5366
broker_display_name                   5366


In [3]:
from collections import Counter, defaultdict
import re


# ============================================================
# QA CONFIGURATION
# ============================================================

VALUE_COUNT_FIELDS = [
    "price_per",
    "furnished",
    "category",
    "listed_by",
    "package_type",
    "sub_category_1"
]

MAX_EXAMPLES = 10


# ============================================================
# URL VALIDATION
# ============================================================

def is_valid_url(value):

    if not isinstance(value, str):
        return False

    value = value.strip()

    if not value:
        return False

    return bool(
        re.match(r"^https?://[^\s]+$", value, re.IGNORECASE)
    )


# ============================================================
# VARIABLES
# ============================================================

total_records = 0
all_columns = set()

# Field presence / null / empty
column_present_count = Counter()
column_null_count = Counter()
column_empty_count = Counter()

# Value counts
value_counts = {
    field: Counter()
    for field in VALUE_COUNT_FIELDS
}

# Invalid URL
invalid_url_count = 0
invalid_url_examples = []

# Broker comparison
broker_mismatch_count = 0
broker_mismatch_examples = []

# Broker uppercase
broker_not_capital_count = 0
broker_not_capital_examples = []

# Spaces
space_issue_count = Counter()
space_examples = defaultdict(list)


# ============================================================
# PROCESS FILE
# ============================================================

print("Starting QA...")
print()

for record_number, record in enumerate(
    stream_json_objects(file_path),
    start=1
):

    total_records += 1

    if not isinstance(record, dict):
        continue


    # ========================================================
    # COLUMNS / SHAPE
    # ========================================================

    all_columns.update(record.keys())

    for column in record:
        column_present_count[column] += 1


    # ========================================================
    # NULL / EMPTY VALUES
    # ========================================================

    for column, value in record.items():

        if value is None:

            column_null_count[column] += 1

        elif isinstance(value, str) and value.strip() == "":

            column_empty_count[column] += 1


    # ========================================================
    # VALUE COUNTS
    # ========================================================

    for field in VALUE_COUNT_FIELDS:

        value = record.get(field)

        if value is None:

            value = "<NULL>"

        elif isinstance(value, str):

            if value.strip() == "":
                value = "<EMPTY>"
            else:
                value = value.strip()

        else:

            value = str(value)

        value_counts[field][value] += 1


    # ========================================================
    # INVALID URL
    # ========================================================

    url = record.get("url")

    if not is_valid_url(url):

        invalid_url_count += 1

        if len(invalid_url_examples) < MAX_EXAMPLES:

            invalid_url_examples.append({
                "record_number": record_number,
                "id": record.get("id", ""),
                "url": url
            })


    # ========================================================
    # BROKER vs BROKER_DISPLAY_NAME
    # ========================================================

    broker = record.get("broker")
    broker_display_name = record.get(
        "broker_display_name"
    )

    broker_str = (
        ""
        if broker is None
        else str(broker).strip()
    )

    broker_display_str = (
        ""
        if broker_display_name is None
        else str(broker_display_name).strip()
    )

    if broker_str and broker_display_str:

        if broker_str != broker_display_str:

            broker_mismatch_count += 1

            if len(broker_mismatch_examples) < MAX_EXAMPLES:

                broker_mismatch_examples.append({
                    "record_number": record_number,
                    "id": record.get("id", ""),
                    "url": record.get("url", ""),
                    "broker": broker,
                    "broker_display_name": broker_display_name
                })


    # ========================================================
    # BROKER CAPITALIZATION
    # ========================================================

    if broker_str:

        if broker_str != broker_str.upper():

            broker_not_capital_count += 1

            if len(broker_not_capital_examples) < MAX_EXAMPLES:

                broker_not_capital_examples.append({
                    "record_number": record_number,
                    "id": record.get("id", ""),
                    "url": record.get("url", ""),
                    "broker": broker
                })


    # ========================================================
    # UNWANTED LEADING / TRAILING SPACES
    # ========================================================

    for field, value in record.items():

        if not isinstance(value, str):
            continue

        if value != value.strip():

            space_issue_count[field] += 1

            if len(space_examples[field]) < MAX_EXAMPLES:

                space_examples[field].append({
                    "record_number": record_number,
                    "id": record.get("id", ""),
                    "url": record.get("url", ""),
                    "value": value
                })


    # ========================================================
    # PROGRESS
    # ========================================================

    if total_records % 100000 == 0:

        print(
            f"Processed: {total_records:,} records"
        )


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 80)
print("                         QA RESULT")
print("=" * 80)


# ============================================================
# 1. SHAPE
# ============================================================

print()
print("-" * 80)
print("1. DATA SHAPE")
print("-" * 80)

print(f"Rows    : {total_records:,}")
print(f"Columns : {len(all_columns):,}")
print(
    f"Shape   : ({total_records:,}, {len(all_columns):,})"
)


# ============================================================
# 2. COMPLETELY EMPTY COLUMNS
# ============================================================

print()
print("-" * 80)
print("2. COMPLETELY EMPTY COLUMNS")
print("-" * 80)

completely_empty = []

for column in sorted(all_columns):

    present = column_present_count[column]
    nulls = column_null_count[column]
    empty = column_empty_count[column]

    non_empty = present - nulls - empty

    if non_empty == 0:

        completely_empty.append(column)


if completely_empty:

    for column in completely_empty:
        print(column)

    print()
    print(
        f"Total completely empty columns: "
        f"{len(completely_empty)}"
    )

else:

    print("No completely empty columns found.")


# ============================================================
# 3. MISSING / NULL / EMPTY VALUES
# ============================================================

print()
print("-" * 80)
print("3. MISSING / NULL / EMPTY VALUES PER COLUMN")
print("-" * 80)

print(
    f"{'Column':<35}"
    f"{'Missing':>12}"
    f"{'NULL':>12}"
    f"{'Empty':>12}"
)

print("-" * 71)

for column in sorted(all_columns):

    present = column_present_count[column]

    missing = total_records - present

    nulls = column_null_count[column]

    empty = column_empty_count[column]

    print(
        f"{column:<35}"
        f"{missing:>12,}"
        f"{nulls:>12,}"
        f"{empty:>12,}"
    )


# ============================================================
# 4. INVALID URL
# ============================================================

print()
print("-" * 80)
print("4. INVALID URL")
print("-" * 80)

print(
    f"Total invalid URLs: {invalid_url_count:,}"
)

print()
print("First 10 invalid URLs")
print("-" * 80)

if invalid_url_examples:

    for item in invalid_url_examples:

        print(
            f"Record Number : {item['record_number']}"
        )
        print(
            f"ID            : {item['id']}"
        )
        print(
            f"URL           : {item['url']}"
        )
        print("-" * 80)

else:

    print("No invalid URLs found.")


# ============================================================
# 5. BROKER vs BROKER_DISPLAY_NAME
# ============================================================

print()
print("-" * 80)
print("5. BROKER vs BROKER_DISPLAY_NAME")
print("-" * 80)

print(
    f"Total mismatches: {broker_mismatch_count:,}"
)

if broker_mismatch_examples:

    print()
    print("First 10 mismatches")
    print("-" * 80)

    for item in broker_mismatch_examples:

        print(
            f"Record Number       : {item['record_number']}"
        )
        print(
            f"ID                  : {item['id']}"
        )
        print(
            f"URL                 : {item['url']}"
        )
        print(
            f"broker              : {item['broker']}"
        )
        print(
            f"broker_display_name : {item['broker_display_name']}"
        )
        print("-" * 80)

else:

    print("No mismatches found.")


# ============================================================
# 6. BROKER CAPITALIZATION
# ============================================================

print()
print("-" * 80)
print("6. BROKER CAPITALIZATION")
print("-" * 80)

print(
    f"Broker values not in CAPITALS: "
    f"{broker_not_capital_count:,}"
)

if broker_not_capital_examples:

    print()
    print("First 10 examples")
    print("-" * 80)

    for item in broker_not_capital_examples:

        print(
            f"Record Number : {item['record_number']}"
        )
        print(
            f"ID            : {item['id']}"
        )
        print(
            f"URL           : {item['url']}"
        )
        print(
            f"broker        : {item['broker']}"
        )
        print("-" * 80)

else:

    print("All non-empty broker values are in CAPITALS.")


# ============================================================
# 7. VALUE COUNTS
# ============================================================

print()
print("-" * 80)
print("7. VALUE COUNTS")
print("-" * 80)

for field in VALUE_COUNT_FIELDS:

    print()
    print(f"### {field}")

    for value, count in value_counts[field].most_common():

        print(
            f"{str(value):<40} : {count:,}"
        )


# ============================================================
# 8. UNWANTED SPACES
# ============================================================

print()
print("-" * 80)
print("8. LEADING / TRAILING SPACES")
print("-" * 80)

total_space_issues = sum(
    space_issue_count.values()
)

print(
    f"Total values with leading/trailing spaces: "
    f"{total_space_issues:,}"
)

if space_issue_count:

    print()
    print("Issue count by column:")

    for field, count in sorted(
        space_issue_count.items(),
        key=lambda x: x[1],
        reverse=True
    ):

        print(
            f"{field:<35} : {count:,}"
        )

    print()
    print("First 10 examples per affected column")

    for field, examples in space_examples.items():

        if not examples:
            continue

        print()
        print(f"### {field}")
        print("-" * 80)

        for item in examples:

            print(
                f"Record Number : {item['record_number']}"
            )
            print(
                f"ID            : {item['id']}"
            )
            print(
                f"URL           : {item['url']}"
            )
            print(
                f"Value         : {repr(item['value'])}"
            )
            print()

else:

    print("No leading/trailing spaces found.")


# ============================================================
# DONE
# ============================================================

print()
print("=" * 80)
print("QA COMPLETED")
print("=" * 80)

Starting QA...

Processed: 100,000 records
Processed: 200,000 records

                         QA RESULT

--------------------------------------------------------------------------------
1. DATA SHAPE
--------------------------------------------------------------------------------
Rows    : 240,121
Columns : 35
Shape   : (240,121, 35)

--------------------------------------------------------------------------------
2. COMPLETELY EMPTY COLUMNS
--------------------------------------------------------------------------------
depth
dtcm_licence
rera_permit_number
sub_category_2
user_id

Total completely empty columns: 5

--------------------------------------------------------------------------------
3. MISSING / NULL / EMPTY VALUES PER COLUMN
--------------------------------------------------------------------------------
Column                                  Missing        NULL       Empty
-----------------------------------------------------------------------
agent_name              

In [4]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [6]:
import json
import re
import pandas as pd

file_path = "/home/user/Downloads/dubizzle_uae_2026_11_01.json"


def stream_json_objects(file_path):
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        buffer = ""

        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            buffer += chunk

            while True:
                buffer = buffer.lstrip()

                if not buffer:
                    break

                try:
                    record, index = decoder.raw_decode(buffer)
                    yield record
                    buffer = buffer[index:]

                except json.JSONDecodeError:
                    break


# Check whitespace issues
issues = {}

for record_no, record in enumerate(
    stream_json_objects(file_path), start=1
):
    if not isinstance(record, dict):
        continue

    for field, value in record.items():

        # Keep only one example per field
        if field in issues or not isinstance(value, str):
            continue

        # Leading, trailing, multiple consecutive spaces,
        # tabs, newlines, and other repeated whitespace
        if (
            value != value.strip()
            or re.search(r"\s{2,}", value)
        ):
            issues[field] = {
                "record_no": record_no,
                "field": field,
                "url": record.get("url", "")
            }

    if record_no % 100000 == 0:
        print(f"Processed {record_no:,} records")


# Display results
result = pd.DataFrame(
    issues.values(),
    columns=["record_no", "field", "url"]
)

print(f"\nFields with whitespace issues: {len(result)}")

if not result.empty:
    display(result)
else:
    print("✓ No whitespace issues found.")

Processed 100,000 records
Processed 200,000 records
Processed 300,000 records
Processed 400,000 records
Processed 500,000 records

Fields with whitespace issues: 6


,record_no,field,url
0,2,description,https://sharjah.dubizzle.com/property-for-rent/rooms-for-rent-flatmates/apartmentflat/2026/10/22/family-room-available-in-al-nahda-sharjah-12-886/
1,12,title,https://sharjah.dubizzle.com/property-for-rent/rooms-for-rent-flatmates/apartmentflat/2026/8/25/fully-furnished-room-for-rent-female-tenan-12-906/
2,4263,reference_number,https://ajman.dubizzle.com/property-for-rent/rooms-for-rent-flatmates/apartmentflat/2026/10/17/furnished-room-for-rent-14-834/
3,5041,agent_name,https://sharjah.dubizzle.com/property-for-sale/commercial/office/2026/4/6/high-value-office-space-in-aljada-cbd-s-12-342606/
4,5366,broker,https://dubai.dubizzle.com/property-for-sale/commercial/office/2026/10/5/canal-view-vacant-close-to-metro-2-101536/
5,5366,broker_display_name,https://dubai.dubizzle.com/property-for-sale/commercial/office/2026/10/5/canal-view-vacant-close-to-metro-2-101536/


In [3]:
from collections import Counter


# ============================================================
# FIELDS TO CHECK
# ============================================================

CHECK_FIELDS = [
    "currency",
    "date",
    "scraped_ts",
    "published_at",
    "iteration_number",
    "last_update"
]

MAX_EXAMPLES = 10


# ============================================================
# COUNTERS
# ============================================================

value_counts = {
    field: Counter()
    for field in CHECK_FIELDS
}

total_records = 0


# ============================================================
# READ FILE
# ============================================================

print("Checking values...")
print()

for record_number, record in enumerate(
    stream_json_objects(file_path),
    start=1
):

    total_records += 1

    if not isinstance(record, dict):
        continue

    for field in CHECK_FIELDS:

        value = record.get(field)

        # Treat missing/null/empty separately
        if value is None:

            value = "<NULL>"

        elif isinstance(value, str):

            value = value.strip()

            if value == "":
                value = "<EMPTY>"

        else:

            value = str(value)

        value_counts[field][value] += 1


    # Progress
    if total_records % 100000 == 0:

        print(
            f"Processed: {total_records:,} records"
        )


# ============================================================
# RESULTS
# ============================================================

print()
print("=" * 80)
print("VALUE COUNTS + UNIQUENESS CHECK")
print("=" * 80)


for field in CHECK_FIELDS:

    counts = value_counts[field]

    total_values = sum(counts.values())

    unique_values = len(counts)

    duplicate_values = sum(
        1
        for count in counts.values()
        if count > 1
    )

    duplicate_records = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )


    # --------------------------------------------------------
    # FIELD HEADER
    # --------------------------------------------------------

    print()
    print("-" * 80)
    print(f"FIELD: {field}")
    print("-" * 80)

    print(
        f"Total records       : {total_values:,}"
    )

    print(
        f"Unique values       : {unique_values:,}"
    )

    print(
        f"Duplicate values    : {duplicate_values:,}"
    )

    print(
        f"Duplicate records   : {duplicate_records:,}"
    )


    # --------------------------------------------------------
    # UNIQUENESS
    # --------------------------------------------------------

    if duplicate_values == 0:

        print("Unique?             : YES")

    else:

        print("Unique?             : NO")


    # --------------------------------------------------------
    # VALUE COUNTS
    # --------------------------------------------------------

    print()
    print("Value counts:")

    for value, count in counts.most_common():

        print(
            f"  {str(value):<35} : {count:,}"
        )


    # --------------------------------------------------------
    # DUPLICATE VALUES
    # --------------------------------------------------------

    duplicates = [
        (value, count)
        for value, count in counts.items()
        if count > 1
    ]

    if duplicates:

        print()
        print("First 10 duplicate values:")

        for value, count in sorted(
            duplicates,
            key=lambda x: x[1],
            reverse=True
        )[:MAX_EXAMPLES]:

            print(
                f"  {str(value):<35} : "
                f"{count:,} occurrences"
            )


# ============================================================
# COMPLETED
# ============================================================

print()
print("=" * 80)
print("CHECK COMPLETED")
print("=" * 80)

Checking values...

Processed: 100,000 records
Processed: 200,000 records
Processed: 300,000 records
Processed: 400,000 records
Processed: 500,000 records
Processed: 600,000 records

VALUE COUNTS + UNIQUENESS CHECK

--------------------------------------------------------------------------------
FIELD: currency
--------------------------------------------------------------------------------
Total records       : 617,511
Unique values       : 1
Duplicate values    : 1
Duplicate records   : 617,510
Unique?             : NO

Value counts:
  AED                                 : 617,511

First 10 duplicate values:
  AED                                 : 617,511 occurrences

--------------------------------------------------------------------------------
FIELD: date
--------------------------------------------------------------------------------
Total records       : 617,511
Unique values       : 1
Duplicate values    : 1
Duplicate records   : 617,510
Unique?             : NO

Value counts:

In [4]:
import sqlite3
import hashlib
import json
import tempfile
import os


# ============================================================
# SETTINGS
# ============================================================

MAX_EXAMPLES = 10


# ============================================================
# TEMPORARY DATABASE
# ============================================================

temp_db = tempfile.NamedTemporaryFile(
    suffix=".db",
    delete=False
)

db_path = temp_db.name
temp_db.close()


conn = sqlite3.connect(db_path)

cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE records (
        record_hash TEXT PRIMARY KEY,
        count INTEGER
    )
""")

conn.commit()


# ============================================================
# VARIABLES
# ============================================================

total_records = 0
unique_records = 0
duplicate_records = 0

duplicate_examples = []


# ============================================================
# PROCESS JSON
# ============================================================

print("=" * 80)
print("CHECKING COMPLETE RECORD UNIQUENESS")
print("=" * 80)
print()

for record_number, record in enumerate(
    stream_json_objects(file_path),
    start=1
):

    total_records += 1

    # --------------------------------------------------------
    # Create a consistent representation of the entire record
    # --------------------------------------------------------

    record_string = json.dumps(
        record,
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":")
    )

    # --------------------------------------------------------
    # Create hash
    # --------------------------------------------------------

    record_hash = hashlib.sha256(
        record_string.encode("utf-8")
    ).hexdigest()


    # --------------------------------------------------------
    # Check whether this complete record already exists
    # --------------------------------------------------------

    cursor.execute(
        """
        SELECT count
        FROM records
        WHERE record_hash = ?
        """,
        (record_hash,)
    )

    result = cursor.fetchone()


    if result is None:

        # Completely new record
        cursor.execute(
            """
            INSERT INTO records
            (record_hash, count)
            VALUES (?, 1)
            """,
            (record_hash,)
        )

        unique_records += 1

    else:

        # Duplicate complete record
        previous_count = result[0]

        cursor.execute(
            """
            UPDATE records
            SET count = count + 1
            WHERE record_hash = ?
            """,
            (record_hash,)
        )

        duplicate_records += 1

        # Keep only first 10 examples
        if len(duplicate_examples) < MAX_EXAMPLES:

            duplicate_examples.append({
                "record_number": record_number,
                "id": record.get("id", ""),
                "url": record.get("url", ""),
                "duplicate_occurrence": previous_count + 1
            })


    # --------------------------------------------------------
    # Commit periodically
    # --------------------------------------------------------

    if total_records % 10000 == 0:

        conn.commit()


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if total_records % 100000 == 0:

        print(
            f"Processed: {total_records:,} records"
        )


# Final commit
conn.commit()


# ============================================================
# FINAL RESULT
# ============================================================

cursor.execute(
    "SELECT COUNT(*) FROM records"
)

unique_hashes = cursor.fetchone()[0]


print()
print("=" * 80)
print("COMPLETE UNIQUENESS RESULT")
print("=" * 80)

print(
    f"Total records          : {total_records:,}"
)

print(
    f"Unique complete records: {unique_hashes:,}"
)

print(
    f"Duplicate records      : {duplicate_records:,}"
)


# ============================================================
# UNIQUE / NOT UNIQUE
# ============================================================

if duplicate_records == 0:

    print()
    print("RESULT: ALL RECORDS ARE COMPLETELY UNIQUE.")

else:

    print()
    print("RESULT: DUPLICATE COMPLETE RECORDS FOUND.")


# ============================================================
# DUPLICATE EXAMPLES
# ============================================================

print()
print("-" * 80)
print("FIRST 10 COMPLETE DUPLICATE RECORDS")
print("-" * 80)

if duplicate_examples:

    for item in duplicate_examples:

        print()
        print(
            f"Record Number       : "
            f"{item['record_number']}"
        )

        print(
            f"ID                  : "
            f"{item['id']}"
        )

        print(
            f"URL                 : "
            f"{item['url']}"
        )

        print(
            f"Occurrence          : "
            f"{item['duplicate_occurrence']}"
        )

else:

    print("No completely duplicate records found.")


# ============================================================
# CLOSE DATABASE
# ============================================================

conn.close()

# Delete temporary database
os.remove(db_path)

print()
print("=" * 80)
print("CHECK COMPLETED")
print("=" * 80)

CHECKING COMPLETE RECORD UNIQUENESS

Processed: 100,000 records
Processed: 200,000 records
Processed: 300,000 records
Processed: 400,000 records
Processed: 500,000 records
Processed: 600,000 records

COMPLETE UNIQUENESS RESULT
Total records          : 617,511
Unique complete records: 617,511
Duplicate records      : 0

RESULT: ALL RECORDS ARE COMPLETELY UNIQUE.

--------------------------------------------------------------------------------
FIRST 10 COMPLETE DUPLICATE RECORDS
--------------------------------------------------------------------------------
No completely duplicate records found.

CHECK COMPLETED
